In [1]:
import pandas as pd

In [ ]:
# Load
df = pd.read_csv('./outputs/Global_Graphite_Projects.csv')
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

In [ ]:
# 1. Merge Name columns
df['Name'] = df['Facility Name'].fillna(df['Site Name']).fillna(df['Deposit Name'])

# 2. Merge Commodity: fill blanks from Commodity 1
df['Commodity'] = df['Commodity'].fillna(df['Commodity 1'])

# Standardise Annual Production Capacity to metric tons
unit_to_mt = {
    'mt':                   1,          # metric tons → already mt
    'Metric tons':          1,          # Africa style
    'kmt':                  1_000,      # thousand metric tons (China/IndoPac)
    'Thousand metric tons': 1_000,      # SW Asia style
    'tmt':                  1_000,      # thousand metric tons (LAC style)
}

mask = df['Capacity Units'].notna() & df['Annual Production Capacity'].notna()
df.loc[mask, 'Annual Production Capacity'] = df.loc[mask].apply(
    lambda r: r['Annual Production Capacity'] * unit_to_mt.get(r['Capacity Units'], 1), axis=1
)
df.loc[mask, 'Capacity Units'] = 'mt'

# Flag any units we didn't handle
unknown_units = df.loc[mask & ~df['Capacity Units'].isin(['mt']), 'Capacity Units'].unique()
if len(unknown_units):
    print(f"WARNING: unhandled capacity units: {unknown_units}")

# Merge Status columns
df['Status'] = (df['Facility Status']
    .fillna(df['Exploration Status'])
    .fillna(df['Operating Status'])
    .fillna(df['Development Status']))

# 4. Merge Facility Type / Feature Type
df['Facility Type'] = df['Facility Type'].fillna(df['Feature Type'])

# 5. Merge USGS ID columns
df['USGS ID'] = (df['USGS Facility ID']
    .fillna(df['Deposit ID'])
    .fillna(df['Exploration ID'])
    .fillna(df['Development ID']))

# 6. Merge Owner columns
df['Owner'] = df['Equity Owner 1'].fillna(df['Owner Name'])

# 7. Merge Location Accuracy / Location Confidence
df['Location Accuracy'] = df['Location Accuracy'].fillna(df['Location Confidence'])

# 8. Merge Alternate Name(s) / Alternate Name
df['Alternate Name(s)'] = df['Alternate Name(s)'].fillna(df['Alternate Name'])

# 9. Merge ADM1 Name / AMD1 Name (China typo)
df['ADM1 Name'] = df['ADM1 Name'].fillna(df['AMD1 Name'])

# 10. Merge Data Source / Mineral Facility Data Source
df['Data Source'] = df['Data Source'].fillna(df['Mineral Facility Data Source'])

# 11. Merge Location Description and Location Notes
# These contain complementary info (admin region vs distance description),
# so concatenate rather than replace
def merge_location_text(row):
    desc = row['Location Description']
    notes = row['Location Notes']
    parts = [str(x) for x in [desc, notes] if pd.notna(x)]
    return '; '.join(parts) if parts else None

df['Location Description'] = df.apply(merge_location_text, axis=1)

# 11b. Merge Year / Source Year / Project Active into Reference Year
# Year = edition year of USGS Minerals Yearbook (data snapshot year)
# Source Year = publication year of most recent source material used
# Project Active = last year the project was verified as active
# All indicate data recency. merged in order of specificity
df['Reference Year'] = df['Year'].fillna(df['Source Year']).fillna(df['Project Active'])

# 12. Drop the columns that were merged into others
drop_cols = [
    # Merged into new columns
    'Facility Name', 'Site Name', 'Deposit Name',           # → Name
    'Facility Status', 'Exploration Status',                # → Status
    'Operating Status', 'Development Status',               # → Status
    'Feature Type',                                         # → Facility Type
    'USGS Facility ID', 'Deposit ID',                       # → USGS ID
    'Exploration ID', 'Development ID',                     # → USGS ID
    'Commodity 1',                                          # → Commodity
    'Owner Name', 'Equity Owner 1',                         # → Owner
    'Location Confidence',                                  # → Location Accuracy
    'Alternate Name',                                       # → Alternate Name(s)
    'AMD1 Name',                                            # → ADM1 Name
    'Mineral Facility Data Source',                         # → Data Source
    'Location Notes',                                       # → Location Description
    # Empty or useless
    'DsgAttr9',                                             # 0 non-null
    'geometry',                                             # Text repr of geometry, not useful in CSV
]
# Only drop columns that actually exist
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# 13. Reorder columns: important first, sources/IDs at the back
front_cols = [
    'Region', 'Source_Layer', 'Country (Short Form)', 'Name', 'Alternate Name(s)',
    'USGS ID', 'Commodity Type', 'Commodity', 'Commodity Product',
    'Status', 'Facility Type', 'Deposit Type',
    'Annual Production Capacity', 'Capacity Units', 'Capacity Notes',
    'Major Operating Company', 'Owner',
    'Equity Owner 2', 'Equity Owner 3', 'Equity Owner 4',
    'ADM1 Name', 'Location Description', 'DD Latitude', 'DD Longitude',
    'Location Accuracy',
]

back_cols = [
    # Secondary commodity columns
    'Commodity 2', 'Commodity 3', 'Commodity 4', 'Commodity 5',
    'Commodity 6', 'Commodity 7', 'Commodity 8', 'Commodity 9',
    'Commodity 10', 'Commodity 11',
    # Metadata & source fields
    'Label', 'Multiple Commodities', 'Multiple Products', 'Product Notes',
    'Year', 'Source Year', 'Project Active',
    'Shared Capacity', 'Estimated Capacity', 'Facility Comments',
    'Critical Commodity', 'USGS MRData ID', 'USGS WMED ID',
    'Owner Type', 'Owner Share', 'Equity Owner 5', 'Equity Owner 6',
    'Data Source', 'GIS Data Source',
    'Location Source 1', 'Location Source 2',
    'Operation Source 1', 'Operation Source 2',
    'Operating Stage Source 1', 'Operating Stage Source 2',
    'Company Source 1', 'Company Source 2',
    'Commodity Source 1', 'Commodity Source 2',
    'Capacity Source 1', 'Capacity Source 2',
    'Other Source', 'Source notes', 'Source ID',
    'ROWID', 'ISO_CC', 'FACID_NUM',
]

# Build final column order
front = [c for c in front_cols if c in df.columns]
back = [c for c in back_cols if c in df.columns]
middle = [c for c in df.columns if c not in front and c not in back]
df = df[front + middle + back]

print(f"Output: {len(df)} rows, {len(df.columns)} columns")
print(f"Columns: {df.columns.tolist()}")

# Save
df.to_csv('./outputs/Global_Graphite_Projects_clean.csv', index=False)
print("Saved to Global_Graphite_Projects_clean.csv")

Loaded 183 rows, 90 columns
Output: 183 rows, 73 columns
Columns: ['Region', 'Source_Layer', 'Country (Short Form)', 'Name', 'Alternate Name(s)', 'USGS ID', 'Commodity Type', 'Commodity', 'Commodity Product', 'Status', 'Facility Type', 'Deposit Type', 'Annual Production Capacity', 'Capacity Units', 'Capacity Notes', 'Major Operating Company', 'Owner', 'Equity Owner 2', 'Equity Owner 3', 'Equity Owner 4', 'ADM1 Name', 'Location Description', 'DD Latitude', 'DD Longitude', 'Location Accuracy', 'Reference Year', 'Commodity 2', 'Commodity 3', 'Commodity 4', 'Commodity 5', 'Commodity 6', 'Commodity 7', 'Commodity 8', 'Commodity 9', 'Commodity 10', 'Commodity 11', 'Label', 'Multiple Commodities', 'Multiple Products', 'Product Notes', 'Year', 'Source Year', 'Project Active', 'Shared Capacity', 'Estimated Capacity', 'Facility Comments', 'Critical Commodity', 'USGS MRData ID', 'USGS WMED ID', 'Owner Type', 'Owner Share', 'Equity Owner 5', 'Equity Owner 6', 'Data Source', 'GIS Data Source', 'Loc